# DeepAlgPro data exploration by split

This notebook keeps DeepAlgPro `train` and `test` separate at every step. Each split is parsed from FASTA into a tabular dataset with `sequence_id`, `sequence`, and `label`, checked for duplicates and class conflicts, filtered against the curated split CSVs produced in notebook `01`, downsampled within the split, visualized, and saved to its own cleaned CSV.

In [ ]:
import os
from pathlib import Path

MPLCONFIGDIR = Path.cwd().resolve() / ".matplotlib"
MPLCONFIGDIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import matplotlib.pyplot as plt
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing a data/ directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data"
SPLIT_FASTA_PATHS = {
    "train": DATA_DIR / "deepalgpro_all.train.fasta",
    "test": DATA_DIR / "deepalgpro_all.test.fasta",
}
SPLIT_OUTPUT_PATHS = {
    "train": DATA_DIR / "deepalgpro_train_cleaned.csv",
    "test": DATA_DIR / "deepalgpro_test_cleaned.csv",
}
OVERLAP_PATHS = {
    "positives_splitA": DATA_DIR / "positives_splitA.csv",
    "positives_splitB": DATA_DIR / "positives_splitB.csv",
    "negatives_splitA": DATA_DIR / "negatives_splitA.csv",
    "negatives_splitB": DATA_DIR / "negatives_splitB.csv",
}

for split_name, fasta_path in SPLIT_FASTA_PATHS.items():
    if not fasta_path.exists():
        raise FileNotFoundError(f"Missing {split_name} FASTA file: {fasta_path}")

for source_name, overlap_path in OVERLAP_PATHS.items():
    if not overlap_path.exists():
        raise FileNotFoundError(f"Missing overlap reference for {source_name}: {overlap_path}")

print(f"Project root: {PROJECT_ROOT}")
print("Split FASTA files:")
for split_name, fasta_path in SPLIT_FASTA_PATHS.items():
    print(f"  - {split_name}: {fasta_path.name}")


In [ ]:
def parse_fasta(fasta_path: Path) -> list[dict]:
    records = []
    sequence_id = None
    sequence_lines = []

    with fasta_path.open() as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue

            if line.startswith(">"):
                if sequence_id is not None:
                    sequence = "".join(sequence_lines)
                    records.append(
                        {
                            "sequence_id": sequence_id,
                            "sequence": sequence,
                            "label": 1 if sequence_id.startswith("allergen_") else 0,
                        }
                    )
                sequence_id = line[1:]
                sequence_lines = []
            else:
                sequence_lines.append(line)

    if sequence_id is not None:
        sequence = "".join(sequence_lines)
        records.append(
            {
                "sequence_id": sequence_id,
                "sequence": sequence,
                "label": 1 if sequence_id.startswith("allergen_") else 0,
            }
        )

    return records


split_dfs = {
    split_name: pd.DataFrame(parse_fasta(fasta_path)).reset_index(drop=True)
    for split_name, fasta_path in SPLIT_FASTA_PATHS.items()
}

for split_name, split_df in split_dfs.items():
    print(f"{split_name} preview")
    display(split_df.head())


In [ ]:
for split_name, split_df in split_dfs.items():
    lengths = split_df["sequence"].str.len()
    class_counts = split_df["label"].value_counts().sort_index()
    duplicate_sequences = split_df[split_df.duplicated(subset=["sequence"], keep=False)]["sequence"].nunique()
    cross_class_sequences = (
        split_df.groupby("sequence")["label"]
        .nunique()
        .loc[lambda s: s > 1]
    )

    print(f"=== {split_name.upper()} ===")
    print(f"Total sequences: {len(split_df):,}")
    print("Class balance:")
    print(class_counts.rename(index={0: "non-allergenic", 1: "allergenic"}).to_string())
    print("Sequence length summary:")
    print(f"  min = {lengths.min()}")
    print(f"  max = {lengths.max()}")
    print(f"  mean = {lengths.mean():.2f}")
    print(f"Duplicate sequences: {int(duplicate_sequences):,}")
    print(f"Sequences appearing in both classes: {len(cross_class_sequences):,}")
    print()


In [ ]:
shared_sequences = set(split_dfs["train"]["sequence"]) & set(split_dfs["test"]["sequence"])
print(f"Exact sequence overlap between train and test: {len(shared_sequences):,}")

if shared_sequences:
    train_overlap_rows = split_dfs["train"].loc[split_dfs["train"]["sequence"].isin(shared_sequences), ["sequence_id", "label", "sequence"]]
    test_overlap_rows = split_dfs["test"].loc[split_dfs["test"]["sequence"].isin(shared_sequences), ["sequence_id", "label", "sequence"]]
    print("Train examples with overlap:")
    display(train_overlap_rows.head(10))
    print("Test examples with overlap:")
    display(test_overlap_rows.head(10))


In [ ]:
overlap_sequences_by_source = {}
for source_name, overlap_path in OVERLAP_PATHS.items():
    overlap_df = pd.read_csv(overlap_path)
    if "sequence" not in overlap_df.columns:
        raise KeyError(f"{overlap_path} does not contain a 'sequence' column.")
    overlap_sequences_by_source[source_name] = set(overlap_df["sequence"].dropna().astype(str))
    print(f"Loaded {len(overlap_sequences_by_source[source_name]):,} reference sequences from {source_name}")

cleaned_split_dfs = {}
for split_name, split_df in split_dfs.items():
    combined_mask = pd.Series(False, index=split_df.index)
    print(f"=== {split_name.upper()} OVERLAP REMOVAL ===")
    for source_name, overlap_sequences in overlap_sequences_by_source.items():
        mask = split_df["sequence"].isin(overlap_sequences)
        combined_mask |= mask
        print(f"Removed due to overlap with {source_name}: {int(mask.sum()):,}")
    cleaned_split_dfs[split_name] = split_df.loc[~combined_mask, ["sequence_id", "sequence", "label"]].copy()
    print(f"Total unique rows removed in {split_name}: {int(combined_mask.sum()):,}")
    print(f"Rows remaining in {split_name}: {len(cleaned_split_dfs[split_name]):,}")
    print()


In [ ]:
balanced_split_dfs = cleaned_split_dfs

for split_name, split_df in cleaned_split_dfs.items():
    class_counts = split_df["label"].value_counts()
    n_neg = int(class_counts.get(0, 0))
    n_pos = int(class_counts.get(1, 0))
    total = int(len(split_df))
    pos_weight = n_neg / n_pos if n_pos else float("inf")

    print(f"{split_name}: total={total:,}, neg={n_neg:,}, pos={n_pos:,}")
    print(f"{split_name}: pos_weight = {pos_weight:.6f}" if n_pos else f"{split_name}: pos_weight = inf")
    print()


In [ ]:
for split_name, split_df in cleaned_split_dfs.items():
    output_path = SPLIT_OUTPUT_PATHS[split_name]
    split_df[["sequence_id", "sequence", "label"]].to_csv(output_path, index=False)
    final_counts = split_df["label"].value_counts().sort_index()
    print(f"Saved {split_name} cleaned dataset to: {output_path}")
    print(final_counts.rename(index={0: "non-allergenic", 1: "allergenic"}).to_string())
    print()


In [ ]:
for split in ["train", "test"]:
    df = balanced_split_dfs[split]
    counts = df["label"].value_counts().to_dict()
    
    print(f"\n=== {split.upper()} FINAL DATASET ===")
    print(f"Total sequences: {len(df):,}")
    print(f"Class distribution: {counts}")